# 🔄 Workflow Concepts - Python AI Workflows

This notebook explains the key concepts of building AI workflows using Python with Azure OpenAI.

> 📝 **Hands-on Exercises**: After reviewing these concepts, complete the exercises in [EXERCISES.md](EXERCISES.md).

## Table of Contents

1. [Setup](#setup-install-required-packages)
2. [What is a Workflow?](#1-what-is-a-workflow)
3. [Core Building Blocks](#2-core-building-blocks)
4. [Sequential Workflow](#3-sequential-workflow)
5. [Concurrent Workflow](#4-concurrent-workflow)
6. [Human-in-the-Loop Workflow](#5-human-in-the-loop-workflow)
7. [Best Practices](#6-best-practices)

## Setup: Install Required Packages

Run the following cell to install the required packages for Workflows and Azure OpenAI integration.

In [ ]:
# Setup: Install required packages
# Run this cell first to ensure all dependencies are installed

import subprocess
import sys

packages = [
    "openai",
    "azure-identity",
    "python-dotenv"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    
print("✅ All packages installed successfully!")

In [ ]:
# Configuration: Load environment variables
# Set up your Azure OpenAI connection

import os
import json
from pathlib import Path
from dotenv import load_dotenv


def find_config_path(start_path: str) -> str:
    """Find the 'python' folder by traversing up from start_path."""
    current_dir = Path(start_path)
    
    while current_dir is not None:
        if current_dir.name.lower() == "python":
            return str(current_dir)
        if current_dir.parent == current_dir:
            break
        current_dir = current_dir.parent
    
    # Fallback to start path if python folder not found
    return start_path


def load_env_file(env_path: str) -> dict:
    """Load environment variables from .env file (JSON format)."""
    env_file = Path(env_path) / ".env"
    
    if not env_file.exists():
        return {}
    
    try:
        with open(env_file, 'r') as f:
            content = f.read()
            env_vars = json.loads(content)
            
            # Set environment variables
            for key, value in env_vars.items():
                os.environ[key] = str(value)
            
            return env_vars
    except json.JSONDecodeError:
        # Fallback: try loading as standard dotenv format
        load_dotenv(env_file, override=True)
        return {}
    except IOError as e:
        print(f"Warning: Failed to load .env file: {e}")
        return {}


# Load environment variables from .env file in the python root folder
config_path = find_config_path(os.getcwd())
env_vars = load_env_file(config_path)
if env_vars:
    print(f"✅ Loaded {len(env_vars)} environment variables from: {config_path}/.env")
else:
    # Fallback: try loading from current directory
    load_dotenv()
    print("⚠️ Loaded .env from current directory (fallback)")

# Check required configuration
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

if endpoint:
    print(f"✅ Azure OpenAI Endpoint: {endpoint}")
    print(f"✅ Deployment Name: {deployment}")
else:
    print("⚠️ Azure endpoint not set")
    print("Please set AZURE_OPENAI_ENDPOINT in your .env file.")
    print("Optionally set:")
    print("  - AZURE_OPENAI_DEPLOYMENT_NAME (default: gpt-4o-mini)")
    print("  - AZURE_OPENAI_API_KEY (for API key auth)")

# Check authentication method
api_key = os.getenv("AZURE_OPENAI_API_KEY")
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

if api_key:
    print("✅ Authentication: API Key")
elif tenant_id and client_id and client_secret:
    print("✅ Authentication: Service Principal")
else:
    print("✅ Authentication: Azure CLI / DefaultAzureCredential")

In [ ]:
# Create Azure OpenAI Client
# This demonstrates how to set up the client with different auth methods

import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, AzureCliCredential, ClientSecretCredential, get_bearer_token_provider

# Get configuration
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

if not endpoint:
    raise ValueError("Azure endpoint not set. Set AZURE_OPENAI_ENDPOINT environment variable.")

api_version = "2024-02-15-preview"

# Try authentication methods in order
if api_key:
    client = AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version
    )
    print(f"✅ Created AzureOpenAI client with API Key auth")
    print(f"   Endpoint: {endpoint}")
    print(f"   Deployment: {deployment}")
else:
    # Token-based auth
    print("⚠️ No API key found, using token-based authentication")
    try:
        credential = AzureCliCredential()
        token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
        client = AzureOpenAI(
            azure_endpoint=endpoint,
            azure_ad_token_provider=token_provider,
            api_version=api_version
        )
        print("✅ Created AzureOpenAI client with Azure CLI auth")
    except Exception as e:
        credential = DefaultAzureCredential()
        token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
        client = AzureOpenAI(
            azure_endpoint=endpoint,
            azure_ad_token_provider=token_provider,
            api_version=api_version
        )
        print("✅ Created AzureOpenAI client with DefaultAzureCredential")

In [ ]:
# Define the Support Ticket model
# This is the common data model used across all workflow patterns

from dataclasses import dataclass
from enum import Enum


class TicketPriority(Enum):
    """Ticket priority levels."""
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"
    CRITICAL = "Critical"


@dataclass
class SupportTicket:
    """Represents a customer support ticket."""
    ticket_id: str
    customer_id: str
    customer_name: str
    subject: str
    description: str
    priority: TicketPriority = TicketPriority.MEDIUM


# Example ticket
example_ticket = SupportTicket(
    ticket_id="TICKET-001",
    customer_id="CUST-001",
    customer_name="John Smith",
    subject="Cannot access my account",
    description="I've been trying to log in for the past hour but keep getting an 'invalid credentials' error. I'm sure my password is correct.",
    priority=TicketPriority.HIGH
)

print("✅ SupportTicket model defined")
print()
print("Example Ticket:")
print(f"  ID: {example_ticket.ticket_id}")
print(f"  Customer: {example_ticket.customer_name} ({example_ticket.customer_id})")
print(f"  Priority: {example_ticket.priority.value}")
print(f"  Subject: {example_ticket.subject}")

In [ ]:
# Workflow Building Blocks - Executor Base Class
# This is the foundation for all workflow patterns

from dataclasses import dataclass
from typing import Any


@dataclass
class WorkflowEvent:
    """Event emitted during workflow execution."""
    executor_id: str
    data: Any


class Executor:
    """Base class for workflow executors."""
    
    def __init__(self, name: str):
        self.name = name
    
    async def handle(self, input_data: Any) -> tuple[Any, WorkflowEvent]:
        """Execute the function and return result with event."""
        raise NotImplementedError("Subclasses must implement handle()")


print("✅ Workflow base classes defined")
print()
print("Building Blocks:")
print("  - WorkflowEvent: Tracks execution events")
print("  - Executor: Base class for workflow steps (implement handle() method)")

In [ ]:
# Test the Azure OpenAI connection with a simple workflow
# This verifies everything is working before the hands-on exercises

async def test_ai_call():
    """Test a simple AI call."""
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Say 'Hello from Workflow Lab!' in exactly 5 words."}
        ],
        max_tokens=50
    )
    return response.choices[0].message.content

# Run the test
import asyncio
result = await test_ai_call()

print("Response from Azure OpenAI:")
print(f"  {result}")
print()
print(f"✅ Azure OpenAI connection successful!")
print("You're ready to start the hands-on exercises!")

## 1. What is a Workflow?

A **Workflow** is a series of **Executors** connected in sequence or parallel. Data flows through the workflow, being processed by each executor.

```
┌───────────┐     ┌───────────┐     ┌───────────┐
│ Executor  │────▶│ Executor  │────▶│ Executor  │
│    A      │     │    B      │     │    C      │
└───────────┘     └───────────┘     └───────────┘
```

### Key Benefits

| Benefit | Description |
|---------|-------------|
| **Composability** | Build complex workflows from simple, reusable components |
| **AI Integration** | Seamlessly integrate AI agents into the workflow |
| **Human Oversight** | Pause workflows to get human input or approval |
| **Async Support** | Leverage Python's asyncio for concurrent execution |

## 2. Core Building Blocks

### 2.1 Executors

An **Executor** is a unit of work in a workflow. It receives input, processes it, and returns output via the `handle()` method.

```python
from dataclasses import dataclass
from typing import Any

@dataclass
class WorkflowEvent:
    """Event emitted during workflow execution."""
    executor_id: str
    data: Any


class Executor:
    """Base class for workflow executors."""
    
    def __init__(self, name: str):
        self.name = name
    
    async def handle(self, input_data: Any) -> tuple[Any, WorkflowEvent]:
        """Execute the function and return result with event."""
        raise NotImplementedError
```

### Example: Ticket Intake Executor

```python
class TicketIntakeExecutor(Executor):
    """Executor that handles ticket intake and validation."""
    
    def __init__(self):
        super().__init__("TicketIntake")
    
    async def handle(self, ticket: SupportTicket) -> tuple[str, WorkflowEvent]:
        if not ticket.subject or not ticket.description:
            raise ValueError("Support ticket must have a subject and description.")
        
        ticket_text = f\"\"\"
Ticket ID: {ticket.ticket_id}
Customer ID: {ticket.customer_id}
Customer Name: {ticket.customer_name}
Priority: {ticket.priority.value}
Subject: {ticket.subject}
Description: {ticket.description}
\"\"\"
        event = WorkflowEvent(executor_id=self.name, data=ticket_text)
        return ticket_text, event
```

### 2.2 AI Agents

Integrate AI models (like Azure OpenAI) into workflows. Agents use a `process()` method to call the AI model:

```python
from openai import AzureOpenAI

class CategorizationAgent:
    """AI agent that categorizes support tickets."""
    
    INSTRUCTIONS = \"\"\"
You are a customer support ticket categorization specialist.
Analyze the incoming support ticket and categorize it into:
- BILLING: Payment issues, invoices, subscription
- TECHNICAL: Software bugs, errors, how-to questions
- GENERAL: Account inquiries, feedback

Respond with JSON: {"category": "CATEGORY", "priority": "HIGH|MEDIUM|LOW"}
\"\"\"
    
    def __init__(self, client: AzureOpenAI, deployment: str):
        self.client = client
        self.deployment = deployment
        self.name = "CategorizationAgent"
    
    async def process(self, ticket_text: str) -> str:
        """Categorize the ticket using AI."""
        response = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": self.INSTRUCTIONS},
                {"role": "user", "content": ticket_text}
            ]
        )
        return response.choices[0].message.content
```

### 2.3 Azure OpenAI Client Factory

The client factory supports multiple authentication methods (API Key, Service Principal, Azure CLI, DefaultAzureCredential):

```python
import os
from openai import AzureOpenAI
from azure.identity import AzureCliCredential, ClientSecretCredential, DefaultAzureCredential, get_bearer_token_provider

def create_chat_client() -> AzureOpenAI:
    """Create Azure OpenAI client with multiple authentication options."""
    endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    if not endpoint:
        raise ValueError("AZURE_OPENAI_ENDPOINT is required")
    
    api_version = "2024-02-15-preview"
    
    # Try API Key first
    api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    if api_key:
        return AzureOpenAI(azure_endpoint=endpoint, api_key=api_key, api_version=api_version)
    
    # Try Service Principal
    tenant = os.environ.get("AZURE_TENANT_ID")
    client_id = os.environ.get("AZURE_CLIENT_ID")
    secret = os.environ.get("AZURE_CLIENT_SECRET")
    if tenant and client_id and secret:
        credential = ClientSecretCredential(tenant, client_id, secret)
        token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
        return AzureOpenAI(azure_endpoint=endpoint, azure_ad_token_provider=token_provider, api_version=api_version)
    
    # Try Azure CLI, then DefaultAzureCredential
    try:
        credential = AzureCliCredential()
        token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
        return AzureOpenAI(azure_endpoint=endpoint, azure_ad_token_provider=token_provider, api_version=api_version)
    except Exception:
        credential = DefaultAzureCredential()
        token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
        return AzureOpenAI(azure_endpoint=endpoint, azure_ad_token_provider=token_provider, api_version=api_version)
```

## 3. Sequential Workflow

A **Sequential Workflow** processes data through a linear pipeline where each step depends on the previous one.

### Architecture

```
┌─────────────┐    ┌──────────────────┐    ┌───────────────────┐
│   Ticket    │───▶│  Categorization  │───▶│    Response       │
│   Intake    │    │   AI Agent       │    │    AI Agent       │
└─────────────┘    └──────────────────┘    └───────────────────┘
```

### Implementation

The sequential workflow uses **constructor injection** to wire up the pipeline:

```python
class SequentialWorkflow:
    """Sequential workflow that chains executors together."""
    
    def __init__(
        self,
        ticket_intake: TicketIntakeExecutor,
        categorization_agent: CategorizationAgent,
        categorization_bridge: CategorizationBridgeExecutor,
        response_agent: ResponseAgent,
        response_bridge: ResponseBridgeExecutor,
    ):
        self.ticket_intake = ticket_intake
        self.categorization_agent = categorization_agent
        self.categorization_bridge = categorization_bridge
        self.response_agent = response_agent
        self.response_bridge = response_bridge
    
    async def run(self, ticket: SupportTicket) -> list[WorkflowEvent]:
        """Execute the sequential workflow."""
        events = []
        
        # Step 1: Ticket Intake
        ticket_text, event = await self.ticket_intake.handle(ticket)
        events.append(event)
        
        # Step 2: AI Categorization
        categorization = await self.categorization_agent.process(ticket_text)
        events.append(WorkflowEvent(self.categorization_agent.name, categorization))
        
        # Step 3: Categorization Bridge
        response_prompt, event = await self.categorization_bridge.handle(categorization)
        events.append(event)
        
        # Step 4: AI Response Generation
        response = await self.response_agent.process(response_prompt)
        events.append(WorkflowEvent(self.response_agent.name, response))
        
        # Step 5: Response Bridge (final output)
        final_response, event = await self.response_bridge.handle(response)
        events.append(event)
        
        return events
```

### Usage Example

```python
# Create executors and agents
ticket_intake = TicketIntakeExecutor()
categorization_agent = CategorizationAgent(client, deployment)
categorization_bridge = CategorizationBridgeExecutor()
response_agent = ResponseAgent(client, deployment)
response_bridge = ResponseBridgeExecutor()

# Build the sequential workflow
workflow = SequentialWorkflow(
    ticket_intake=ticket_intake,
    categorization_agent=categorization_agent,
    categorization_bridge=categorization_bridge,
    response_agent=response_agent,
    response_bridge=response_bridge,
)

# Execute
events = await workflow.run(sample_ticket)
```

### Use Cases

- **Document Processing**: Parse → Extract → Summarize → Store
- **Order Processing**: Validate → Calculate → Confirm → Fulfill
- **Support Tickets**: Intake → Categorize → Respond

## 4. Concurrent Workflow

A **Concurrent Workflow** distributes work to multiple agents simultaneously and aggregates results (fan-out/fan-in pattern).

### Architecture

```
                    ┌──────────────────┐
               ┌───▶│  Billing Expert  │───┐
┌──────────┐   │    └──────────────────┘   │    ┌─────────────┐
│ Question │───┤                           ├───▶│  Combined   │
└──────────┘   │    ┌──────────────────┐   │    │  Response   │
               └───▶│ Technical Expert │───┘    └─────────────┘
                    └──────────────────┘
```

### Implementation

```python
import asyncio
from openai import AzureOpenAI


class ChatClientAgent:
    """AI Agent with specific expertise."""
    
    def __init__(self, client: AzureOpenAI, deployment: str, name: str, instructions: str):
        self.client = client
        self.deployment = deployment
        self.name = name
        self.instructions = instructions
    
    async def process(self, message: str) -> str:
        """Process the message and return a response."""
        response = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": message}
            ],
            max_tokens=200
        )
        return response.choices[0].message.content


class ConcurrentWorkflow:
    """Workflow that executes agents concurrently (fan-out/fan-in pattern)."""
    
    def __init__(
        self,
        start_executor: ConcurrentStartExecutor,
        agents: list[ChatClientAgent],
        aggregation_executor: ConcurrentAggregationExecutor,
    ):
        self.start_executor = start_executor
        self.agents = agents
        self.aggregation_executor = aggregation_executor
    
    async def run(self, question: str) -> list[WorkflowEvent]:
        """Execute the concurrent workflow."""
        events = []
        
        # Step 1: Start executor broadcasts the question
        _, start_event = await self.start_executor.handle(question)
        events.append(start_event)
        
        # Step 2: Fan-out - run all agents concurrently
        async def run_agent(agent):
            result = await agent.process(question)
            return agent.name, result
        
        tasks = [run_agent(agent) for agent in self.agents]
        results = await asyncio.gather(*tasks)
        
        # Step 3: Fan-in - aggregate results
        responses = {name: result for name, result in results}
        _, agg_event = await self.aggregation_executor.handle(responses)
        events.append(agg_event)
        
        return events
```

### Usage Example

```python
# Create specialized agents
billing_expert = ChatClientAgent(client, deployment, "BillingExpert",
    "You are an expert in billing and subscription matters.")

technical_expert = ChatClientAgent(client, deployment, "TechnicalExpert",
    "You are an expert in technical support and troubleshooting.")

# Create executors
start = ConcurrentStartExecutor()
aggregation = ConcurrentAggregationExecutor()

# Create and run workflow
workflow = ConcurrentWorkflow(
    start_executor=start,
    agents=[billing_expert, technical_expert],
    aggregation_executor=aggregation,
)
events = await workflow.run("My subscription was charged twice and the app crashes.")
```

### Use Cases

- **Multi-Expert Analysis**: Get perspectives from multiple specialists
- **Parallel Processing**: Process multiple items simultaneously
- **Consensus Building**: Aggregate opinions from multiple AI agents

## 5. Human-in-the-Loop Workflow

A **Human-in-the-Loop Workflow** pauses execution to get human input, approval, or oversight.

### Architecture

```
┌─────────────┐    ┌──────────────┐    ┌────────────────┐    ┌──────────────┐
│   Ticket    │───▶│   AI Draft   │───▶│  Supervisor    │───▶│   Finalize   │
│   Intake    │    │    Agent     │    │   Review       │    │   Response   │
└─────────────┘    └──────────────┘    └────────┬───────┘    └──────────────┘
                                              │
                                              ▼
                                     ┌──────────────┐
                                     │  - Approve   │
                                     │  - Edit      │
                                     │  - Escalate  │
                                     └──────────────┘
```

### Key Components

```python
from enum import Enum
from dataclasses import dataclass
from typing import Optional, Callable

class ReviewAction(Enum):
    APPROVE = "Approve"
    EDIT = "Edit"
    ESCALATE = "Escalate"

@dataclass
class SupervisorReviewRequest:
    """Request sent to supervisor for review."""
    ticket_id: str
    category: str
    priority: str
    draft_response: str

@dataclass
class SupervisorDecision:
    """Supervisor's decision after reviewing the draft."""
    action: ReviewAction
    modified_response: Optional[str]
    notes: str
```

### Implementation

```python
class HumanInTheLoopWorkflow:
    """Workflow that pauses for human review/approval."""
    
    def __init__(
        self,
        ticket_intake: HumanInTheLoopTicketIntakeExecutor,
        draft_agent: DraftAgent,
        draft_bridge: DraftBridgeExecutor,
        finalize: FinalizeExecutor,
    ):
        self.ticket_intake = ticket_intake
        self.draft_agent = draft_agent
        self.draft_bridge = draft_bridge
        self.finalize = finalize
    
    async def run(
        self,
        ticket: SupportTicket,
        supervisor_handler: Callable[[SupervisorReviewRequest], SupervisorDecision],
    ) -> list[WorkflowEvent]:
        """Execute the workflow with human review."""
        events = []
        
        # Step 1: Ticket Intake
        ticket_text, event = await self.ticket_intake.handle(ticket)
        events.append(event)
        
        # Step 2: AI Draft Generation
        draft_response = await self.draft_agent.process(ticket_text)
        events.append(WorkflowEvent(self.draft_agent.name, draft_response))
        
        # Step 3: Create review request
        review_request, event = await self.draft_bridge.handle(draft_response)
        events.append(event)
        
        # Step 4: PAUSE - Get human supervisor decision
        decision = supervisor_handler(review_request)
        events.append(WorkflowEvent("SupervisorReview", decision))
        
        # Step 5: Finalize based on decision
        final_message, event = await self.finalize.handle(decision)
        events.append(event)
        
        return events
```

### Console Supervisor Handler

```python
def handle_supervisor_review(review_request: SupervisorReviewRequest) -> SupervisorDecision:
    """Console-based supervisor review handler."""
    print(f"Ticket: {review_request.ticket_id}")
    print(f"AI Draft: {review_request.draft_response}")
    print("[1] Approve [2] Edit [3] Escalate")
    
    choice = input("Enter choice (1-3): ")
    
    if choice == "1":
        return SupervisorDecision(action=ReviewAction.APPROVE, modified_response=None, notes="Approved as-is")
    elif choice == "2":
        modified = input("Enter modified response: ")
        return SupervisorDecision(action=ReviewAction.EDIT, modified_response=modified, notes="Modified by supervisor")
    else:
        reason = input("Enter escalation reason: ")
        return SupervisorDecision(action=ReviewAction.ESCALATE, modified_response=None, notes=reason)
```

### Use Cases

- **Content Approval**: AI drafts, human approves before publishing
- **Financial Decisions**: AI recommends, human authorizes large transactions
- **Customer Escalation**: AI handles routine, human handles exceptions
- **Quality Control**: AI processes, human spot-checks

## 6. Best Practices

### 6.1 Executor Design

- **Single Responsibility**: Each executor should do one thing well
- **Stateless When Possible**: Avoid storing state in executors
- **Clear Naming**: Use descriptive names that indicate the executor's purpose
- **Type Hints**: Use Python type hints for better code clarity

### 6.2 AI Agent Instructions

```python
# ✅ Good: Specific, structured output
instructions = \"\"\"
Categorize tickets into: BILLING, TECHNICAL, GENERAL
Output JSON: {"category": "CATEGORY", "confidence": 0.0-1.0}
Be concise. Only output the JSON.
\"\"\"

# ❌ Bad: Vague, unstructured
instructions = "Help with customer support"
```

### 6.3 Error Handling

```python
try:
    events = await workflow.run(sample_ticket)
except Exception as e:
    print(f"Workflow error: {e}")
    # Log, retry, or escalate
```

### 6.4 Configuration Management

```python
import os

# Support multiple auth methods via environment variables
# 1. API Key: AZURE_OPENAI_API_KEY
# 2. Azure CLI: (no env vars needed)
# 3. Service Principal: AZURE_TENANT_ID, AZURE_CLIENT_ID, AZURE_CLIENT_SECRET

endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
deployment = os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o-mini")
```

## Summary

| Pattern | When to Use | Key Concept |
|---------|-------------|-------------|
| **Sequential** | Linear processing pipeline | Executor chaining with `handle()` |
| **Concurrent** | Parallel processing, multi-expert | `asyncio.gather()` fan-out/fan-in |
| **Human-in-the-Loop** | Approval, oversight, exceptions | Supervisor handler callback |

### Quick Reference

```python
# Sequential workflow
workflow = SequentialWorkflow(
    ticket_intake=intake, categorization_agent=cat_agent,
    categorization_bridge=cat_bridge, response_agent=resp_agent,
    response_bridge=resp_bridge,
)
events = await workflow.run(sample_ticket)

# Concurrent workflow
workflow = ConcurrentWorkflow(
    start_executor=start, agents=[billing, technical],
    aggregation_executor=aggregation,
)
events = await workflow.run(question)

# Human-in-the-loop workflow
workflow = HumanInTheLoopWorkflow(
    ticket_intake=intake, draft_agent=draft,
    draft_bridge=bridge, finalize=finalize,
)
events = await workflow.run(ticket, handle_supervisor_review)
```